In [1]:
# Cell 1 — Environment setup
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

!pip install -q --upgrade peft trl transformers accelerate bitsandbytes datasets wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 106.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.8/26.8 MB 67.3 MB/s eta 0:00:00


In [2]:
# Cell 2 — Imports and GPU check
import json
import torch
import wandb
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
from accelerate import Accelerator

print(f"CUDA available : {torch.cuda.is_available()}")
print(f"GPU            : {torch.cuda.get_device_name(0)}")

CUDA available : True
GPU            : Tesla T4


In [3]:
# Cell 3 — Config
MODEL_ID   = "mistralai/Mistral-7B-Instruct-v0.2"
OUTPUT_DIR = "edututor-mistral-lora"
HF_REPO    = "Shuvam-Maity/edututor-mistral-lora"

LORA_R         = 8
LORA_ALPHA     = 16
LORA_DROPOUT   = 0.05
TARGET_MODULES = ["q_proj", "v_proj"]

EPOCHS        = 1
BATCH_SIZE    = 1
GRAD_ACCUM    = 8
LEARNING_RATE = 2e-4
MAX_SEQ_LEN   = 256
WARMUP_STEPS  = 30

In [ ]:
# Cell 4 — Login
from huggingface_hub import login

wandb.login("weave_api_key")   
login("hf_token")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: shuvammaity (models-st-xavier-s-college) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
# Cell 5 — Load and format dataset
def format_prompt(sample):
    return {
        "text": (
            f"<s>[INST] {sample['instruction']} [/INST] "
            f"{sample['response']}</s>"    
        )
    }

with open("/data/train.json") as f:
    train_data = json.load(f)

with open("/data/validation.json") as f:
    val_data = json.load(f)

train_dataset = Dataset.from_list(train_data).map(format_prompt)
val_dataset   = Dataset.from_list(val_data).map(format_prompt)

print(f"Train : {len(train_dataset)} samples")
print(f"Val   : {len(val_dataset)} samples")
print(f"\nSample prompt:\n{train_dataset[0]['text'][:300]}")

Map:   0%|          | 0/9205 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Train : 9205 samples
Val   : 1000 samples

Sample prompt:
<s>[INST] What biological agents that infect living hosts contain dna, yet lack the other parts shared by all cells, including a plasma membrane, cytoplasm, and ribosomes? [/INST] viruses. Viruses contain DNA but not much else. They lack the other parts shared by all cells, including a plasma membra


In [6]:
# Cell 6 — Load model in 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

print("Loading Mistral-7B in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    low_cpu_mem_usage=True,
    dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Model loaded. Parameters: {model.num_parameters():,}")
print(f"GPU memory free: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")

Loading Mistral-7B in 4-bit...


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Model loaded. Parameters: 7,241,732,096
GPU memory free: 11.24 GB


In [7]:
# Cell 7 — Attach LoRA adapters
model.config.use_cache = False
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params : {trainable:,}  ({100 * trainable / total:.4f}% of total)")

Trainable params : 3,407,872  (0.0907% of total)


In [8]:
# Cell 8 — Training config
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    fp16=False,
    bf16=True,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    load_best_model_at_end=True,
    report_to="wandb",
    run_name="edututor-mistral-qlora",
    max_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    dataloader_num_workers=0,
)

In [9]:
# Cell 9 — Train
accelerator = Accelerator(mixed_precision="bf16")

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

trainer.args._n_gpu = 1

print(f"Number of GPUs trainer sees: {trainer.args.n_gpu}")
print("Starting training...")
trainer.train()
print("Training complete.")

Adding EOS to train dataset:   0%|          | 0/9205 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/9205 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/9205 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

...

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Number of GPUs trainer sees: 1
Starting training...


wandb: Tracking run with wandb version 0.28.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260702_114708-odbv5bno
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run edututor-mistral-qlora
wandb: ⭐️ View project at https://wandb.ai/models-st-xavier-s-college/huggingface
wandb: 🚀 View run at https://wandb.ai/models-st-xavier-s-college/huggingface/runs/odbv5bno


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
200,1.400607,1.342927,1.374600,160114.000000,0.676434
400,1.340387,1.317495,1.376353,326126.000000,0.681211
600,1.325094,1.305431,1.304444,489681.000000,0.683573
800,1.307818,1.297802,1.307338,653916.000000,0.683823
1000,1.335358,1.292488,1.310722,816423.000000,0.685360
1151,1.308148,1.291870,1.302698,939336.000000,0.685563


Training complete.


In [10]:
# Cell 10 — Save and push to HF Hub
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)

print(f"✅ Pushed to huggingface.co/{HF_REPO}")

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

✅ Pushed to huggingface.co/Shuvam-Maity/edututor-mistral-lora


In [11]:
# Cell 11 — Sanity check
print("Testing fine-tuned model...")

inputs = tokenizer(
    "<s>[INST] What is osmosis? [/INST]",
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.7,
    do_sample=True
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Testing fine-tuned model...
[INST] What is osmosis? [/INST] passive movement. Osmosis is the passive movement of water across a semipermeable membrane. The movement of water across a membrane is determined by the concentration of solute in the solution on either side of the membrane.


In [ ]:
# Cell 12 — Push dataset to HF Hub
from datasets import Dataset as HFDataset

with open("/data/train.json") as f:
    train_data_raw = json.load(f)

hf_dataset = HFDataset.from_list(train_data_raw)
hf_dataset.push_to_hub("Shuvam-Maity/edututor-sciq-instruct")

print("✅ Dataset pushed to huggingface.co/Shuvam-Maity/edututor-sciq-instruct")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Dataset pushed to huggingface.co/Shuvam-Maity/edututor-sciq-instruct
